In [422]:
import numpy as np
import pandas as pd
import seaborn as sns

from helper import convert_to_usd

# scikit-learn imports
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [423]:
complexity_weight_df = pd.read_csv(
    "../data/cleaned/Cleaned Extracted Job Descriptions.csv",
    index_col=0
)

In [424]:
complexity_weight_df.head()

,TITLE,UID,SKILL CATEGORY,SKILL,COMPLEXITY,WEIGHT,COMPLEXITY x WEIGHT
ID,,,,,,,
4,Project Title: Build API access to acquired fl...,4-T1,TECH,Build API access to acquired flood data,4,40,160
4,NaN,4-T3,TECH,Design the API,5,70,350
4,NaN,4-T4,TECH,Optimize the API for efficient data retrieval ...,5,70,350
4,NaN,4-T5,TECH,building APIs,4,40,160
4,NaN,4-T6,TECH,data analysis,4,70,280


In [425]:

salary_df = pd.read_csv("../data/raw/Validators' Skill Sheet - Complexity Score.csv")
salary_df = salary_df.set_index(["ID"])

In [426]:
complexity_weight_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2425 entries, 4 to 8162
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   TITLE                140 non-null    object
 1   UID                  2425 non-null   object
 2   SKILL CATEGORY       2425 non-null   object
 3   SKILL                2425 non-null   object
 4   COMPLEXITY           2425 non-null   int64 
 5   WEIGHT               2425 non-null   int64 
 6   COMPLEXITY x WEIGHT  2425 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 151.6+ KB


In [427]:
freelancer_df = pd.read_csv('../data/raw/Freelancer Data.csv')
# set index to 1
freelancer_df.index = freelancer_df.index + 1

In [428]:
freelancer_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5707 entries, 1 to 5707
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_title              5707 non-null   object 
 1   Industry               6 non-null      object 
 2   Keywords               6 non-null      object 
 3   Assigned Industry      5707 non-null   object 
 4   projectId              5707 non-null   int64  
 5   job_description        5707 non-null   object 
 6   tags                   5707 non-null   object 
 7   client_state           5486 non-null   object 
 8   client_country         5707 non-null   object 
 9   client_average_rating  5707 non-null   float64
 10  client_review_count    5707 non-null   int64  
 11  min_price              5707 non-null   int64  
 12  max_price              5707 non-null   int64  
 13  avg_price              5707 non-null   float64
 14  currency               5707 non-null   object 
 15  rate

In [429]:
freelancer_df.head()

,job_title,Industry,Keywords,Assigned Industry,projectId,job_description,tags,client_state,client_country,client_average_rating,client_review_count,min_price,max_price,avg_price,currency,rate_type
1,development and implementation of a federated ...,Data Science,"data, analytics, python, regression, excel, sp...",AI / ML,37426471,please bid only if you are ready to do the wor...,"['algorithm', 'java', 'python', 'machine learn...",Heilbronn,Germany,5.0,17,8,30,19.0,EUR,fixed
2,Data Scrap,Cybersecurity,"security, SOC, cybersecurity, IoT",Data Science,37400492,I am looking for a freelancer who can help me ...,"['web scraping', 'data mining', 'data entry', ...",Eaubonne,France,5.0,1,30,250,140.0,EUR,fixed
3,Big Data Project,Database / BI,"database, Power BI, SQL, BI, business intellig...",Data Science,37404568,Store Sales Data Analysis: A Data Engineering ...,"['big data sales', 'data science', 'data minin...",Mundra,India,5.0,2,5000,5500,5250.0,INR,fixed
4,Build API access to data,Data Analyst,"data, analytics, analyst, BI, business intelli...",Data Science,37390097,Project Title: Build API access to acquired fl...,"['python', 'php', 'javascript', 'software arch...",Crewe,United Kingdom,5.0,1,250,750,500.0,USD,fixed
5,Develop an AI website from the ground up,Coders/Programmers,"code, coding, program, python, programmer, pro...",AI / ML,37432305,I am looking for a skilled developer to create...,"['java', 'website design', 'html', 'python', '...",Rishon LeTsiyyon,Israel,0.0,0,5000,10000,7500.0,USD,fixed


## Preprocessing

In [430]:
salary_ranges_df = freelancer_df[['min_price', 'max_price', 'job_title']].copy()
merged_salaries_df = salary_ranges_df.merge(salary_df, left_index=True, right_index=True)
merged_salaries_df.head()

,min_price,max_price,job_title,Job Title,Job Description,Sum of Weights,Sum of (Complexity x Weight),Complexity Score,Amount,Currency,Country,Country Category
4,250,750,Build API access to data,Build API access to data,Project Title: Build API access to acquired fl...,840,3710,4.416667,500,USD,United Kingdom,High income
100,30,250,R studio data analysis ranking variables on ho...,Multimodal Neural Network for Financial Data A...,Project Overview: I am looking to develop a cu...,1550,6970,4.496774,500,USD,United Arab Emirates,High income
163,10,20,coding using Rstudio,Reconciliation of GST Invoices between two Sou...,I am looking for a skilled developer who can h...,670,2570,3.835821,7000,INR,India,Lower middle income
194,30,250,Drawing PowerBI dashboard from SQL Server,Need complex sample tableau dashboards (financ...,Project Description: Complex Sample Tableau Da...,1030,4410,4.281553,140,USD,United States,High income
256,8,30,BUS385 Business Analytics for Decision Making ...,App Statistics Module Developer Needed,Project Overview: Seeking an experienced app d...,910,3970,4.362637,500,USD,Denmark,High income


In [431]:
merged_salaries_df.size

1128

In [432]:
merged_salaries_df["Amount"] = merged_salaries_df.apply(convert_to_usd, axis=1, args=["Amount"])

In [433]:
merged_salaries_df.head()

,min_price,max_price,job_title,Job Title,Job Description,Sum of Weights,Sum of (Complexity x Weight),Complexity Score,Amount,Currency,Country,Country Category
4,250,750,Build API access to data,Build API access to data,Project Title: Build API access to acquired fl...,840,3710,4.416667,500.000000,USD,United Kingdom,High income
100,30,250,R studio data analysis ranking variables on ho...,Multimodal Neural Network for Financial Data A...,Project Overview: I am looking to develop a cu...,1550,6970,4.496774,500.000000,USD,United Arab Emirates,High income
163,10,20,coding using Rstudio,Reconciliation of GST Invoices between two Sou...,I am looking for a skilled developer who can h...,670,2570,3.835821,81.628572,INR,India,Lower middle income
194,30,250,Drawing PowerBI dashboard from SQL Server,Need complex sample tableau dashboards (financ...,Project Description: Complex Sample Tableau Da...,1030,4410,4.281553,140.000000,USD,United States,High income
256,8,30,BUS385 Business Analytics for Decision Making ...,App Statistics Module Developer Needed,Project Overview: Seeking an experienced app d...,910,3970,4.362637,500.000000,USD,Denmark,High income


In [434]:
filtered_salary_df = salary_df[[    
    "Complexity Score",
    "Amount",
    "Country",
    "Country Category"
]].copy()

In [435]:
filtered_salary_df.head()

,Complexity Score,Amount,Country,Country Category
ID,,,,
4,4.416667,500,United Kingdom,High income
100,4.496774,500,United Arab Emirates,High income
163,3.835821,7000,India,Lower middle income
194,4.281553,140,United States,High income
256,4.362637,500,Denmark,High income


In [436]:
filtered_complexity_weight_df = complexity_weight_df[[    
    "COMPLEXITY",
    "WEIGHT",
    "SKILL CATEGORY"
]].copy()

In [437]:
filtered_complexity_weight_df['SKILL CATEGORY'].unique()

array(['TECH', 'SOFT SKILL', 'OTHERS'], dtype=object)

In [438]:
filtered_complexity_weight_df.head()

,COMPLEXITY,WEIGHT,SKILL CATEGORY
ID,,,
4,4,40,TECH
4,5,70,TECH
4,5,70,TECH
4,4,40,TECH
4,4,70,TECH


In [439]:
weight_columns = [col for col in filtered_complexity_weight_df.columns if "WEIGHT" in col.upper()]
for col in weight_columns:
    filtered_complexity_weight_df[col] = pd.to_numeric(filtered_complexity_weight_df[col], errors="coerce")
    if filtered_complexity_weight_df[col].max(skipna=True) > 1:
        filtered_complexity_weight_df[col] = filtered_complexity_weight_df[col] / 100
filtered_complexity_weight_df.head()

,COMPLEXITY,WEIGHT,SKILL CATEGORY
ID,,,
4,4,0.4,TECH
4,5,0.7,TECH
4,5,0.7,TECH
4,4,0.4,TECH
4,4,0.7,TECH


generate weights based off complexity scores

In [440]:
# Reconcile complexity and weight using weighted aggregates per index (ID).
sums_df = (
    filtered_complexity_weight_df
    .assign(COMPLEXITY_WEIGHT=lambda df: df["COMPLEXITY"] * df["WEIGHT"])
    .groupby(level=0)
    .agg(
        COMPLEXITY_WEIGHT_SUM=("COMPLEXITY_WEIGHT", "sum"),
        COMPLEXITY_SUM=("COMPLEXITY", "sum"),
        WEIGHT_SUM=("WEIGHT", "sum"),
        ROW_COUNT=("COMPLEXITY_WEIGHT", "size"),
    )
)

# Weighted mean is more faithful than plain row-count averaging.
sums_df["COMPLEXITY_WEIGHT_MEAN"] = (
    sums_df["COMPLEXITY_WEIGHT_SUM"] / sums_df["WEIGHT_SUM"].replace(0, pd.NA)
)

# Log-transform helps stabilize heavy-tailed aggregated magnitudes.
sums_df["LOG_COMPLEXITY_WEIGHT_SUM"] = np.log1p(
    sums_df["COMPLEXITY_WEIGHT_SUM"].clip(lower=0)
)

sums_df.head()

,COMPLEXITY_WEIGHT_SUM,COMPLEXITY_SUM,WEIGHT_SUM,ROW_COUNT,COMPLEXITY_WEIGHT_MEAN,LOG_COMPLEXITY_WEIGHT_SUM
ID,,,,,,
4,37.1,57,8.4,13,4.416667,3.640214
100,69.7,111,15.5,25,4.496774,4.258446
163,25.7,38,6.7,10,3.835821,3.284664
194,44.1,64,10.3,15,4.281553,3.808882
256,39.7,60,9.1,14,4.362637,3.706228


generate three columns for the ratio of tech skills, soft skills, other compared to total number of skills ('ROW_COUNT')

In [441]:
sums_df['ratio_tech_skill'] = filtered_complexity_weight_df['SKILL CATEGORY'].str.contains('tech', case=False, na=False).groupby(level=0).sum() / sums_df['ROW_COUNT']
sums_df['ratio_soft_skill'] = filtered_complexity_weight_df['SKILL CATEGORY'].str.contains('soft', case=False, na=False).groupby(level=0).sum() / sums_df['ROW_COUNT']
sums_df['ratio_other_skill'] = filtered_complexity_weight_df['SKILL CATEGORY'].str.contains('other', case=False, na=False).groupby(level=0).sum() / sums_df['ROW_COUNT']

In [442]:
sums_df.head()

,COMPLEXITY_WEIGHT_SUM,COMPLEXITY_SUM,WEIGHT_SUM,ROW_COUNT,COMPLEXITY_WEIGHT_MEAN,LOG_COMPLEXITY_WEIGHT_SUM,ratio_tech_skill,ratio_soft_skill,ratio_other_skill
ID,,,,,,,,,
4,37.1,57,8.4,13,4.416667,3.640214,0.769231,0.076923,0.153846
100,69.7,111,15.5,25,4.496774,4.258446,0.640000,0.000000,0.360000
163,25.7,38,6.7,10,3.835821,3.284664,0.500000,0.200000,0.300000
194,44.1,64,10.3,15,4.281553,3.808882,0.733333,0.066667,0.200000
256,39.7,60,9.1,14,4.362637,3.706228,0.785714,0.000000,0.214286


In [443]:
left_df = filtered_salary_df.reset_index()
right_df = sums_df.reset_index()

overlap_cols = [col for col in right_df.columns if col in left_df.columns and col != "ID"]
right_df = right_df.drop(columns=overlap_cols)

merged_dataset_raw = (
    left_df
    .merge(right_df, on="ID", how="inner")
    .set_index("ID")
)


In [444]:
merged_dataset_raw.head(10)

,Complexity Score,Amount,Country,Country Category,COMPLEXITY_WEIGHT_SUM,COMPLEXITY_SUM,WEIGHT_SUM,ROW_COUNT,COMPLEXITY_WEIGHT_MEAN,LOG_COMPLEXITY_WEIGHT_SUM,ratio_tech_skill,ratio_soft_skill,ratio_other_skill
ID,,,,,,,,,,,,,
4,4.416667,500,United Kingdom,High income,37.1,57,8.4,13,4.416667,3.640214,0.769231,0.076923,0.153846
100,4.496774,500,United Arab Emirates,High income,69.7,111,15.5,25,4.496774,4.258446,0.640000,0.000000,0.360000
163,3.835821,7000,India,Lower middle income,25.7,38,6.7,10,3.835821,3.284664,0.500000,0.200000,0.300000
194,4.281553,140,United States,High income,44.1,64,10.3,15,4.281553,3.808882,0.733333,0.066667,0.200000
256,4.362637,500,Denmark,High income,39.7,60,9.1,14,4.362637,3.706228,0.785714,0.000000,0.214286
302,4.735849,140,Sri Lanka,Lower middle income,50.2,71,10.6,15,4.735849,3.935740,0.933333,0.000000,0.066667
352,3.953488,20,United States,High income,34.0,51,8.6,13,3.953488,3.555348,0.538462,0.230769,0.230769
411,4.813953,2750,France,High income,62.1,91,12.9,19,4.813953,4.144721,0.789474,0.000000,0.210526
467,4.773810,500,Portugal,High income,40.1,57,8.4,12,4.773810,3.716008,0.916667,0.000000,0.083333


In [445]:
signals = [
    "Amount",
    "Country",
    "Country Category",
    "LOG_COMPLEXITY_WEIGHT_SUM",
    "COMPLEXITY_WEIGHT_MEAN",
    "COMPLEXITY_WEIGHT_SUM",

    "COMPLEXITY_SUM",
    "WEIGHT_SUM",
    "ROW_COUNT",

    "ratio_tech_skill",
    "ratio_other_skill",
    "ratio_soft_skill"
]

data = merged_dataset_raw[signals].copy()

## Modelling

In [ ]:
# Predict amount from weighted-aggregate complexity features + location features.
X = data[[
    "LOG_COMPLEXITY_WEIGHT_SUM",  # this works well, apparently. IDK WHY
    "Country",
    "Country Category",
    "ratio_tech_skill",
    "ratio_other_skill",
    "ratio_soft_skill"
]]
y = np.log1p(data["Amount"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Regression
---

In [447]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01), ["Country", "Country Category"]),
        ("numerical", StandardScaler(), ["LOG_COMPLEXITY_WEIGHT_SUM"])
    ],
    remainder="passthrough",
    sparse_threshold=0,
    verbose_feature_names_out=False,
)

In [448]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [449]:
results = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": root_mean_squared_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred),
}

results

{'MAE': 1.5862073277565543,
 'MSE': 4.696299439288419,
 'RMSE': 2.1670947001200522,
 'R2': 0.42296099063189974}

In [450]:
import optuna
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.model_selection import KFold, cross_val_score

# Predict amount from weighted-aggregate complexity features + location features.
X_opt = X

y_opt = y

X_train_opt, X_test_opt, y_train_opt, y_test_opt = train_test_split(
    X_opt, y_opt, test_size=0.2, random_state=42
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def build_linear_regressor(params):
    model_type = params["model_type"]
    if model_type == "Ridge":
        return Ridge(alpha=params["alpha"])
    if model_type == "Lasso":
        return Lasso(alpha=params["alpha"], max_iter=20000)
    return ElasticNet(

        alpha=params["alpha"],
        l1_ratio=params["l1_ratio"],
        max_iter=20000,
    )

def objective(trial):
    model_type = trial.suggest_categorical("model_type", ["Ridge", "Lasso", "ElasticNet"])
    alpha = trial.suggest_float("alpha", 1e-4, 100.0, log=True)

    params = {
        "model_type": model_type,
        "alpha": alpha,
    }

    if model_type == "ElasticNet":
        params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.05, 0.95)

    preprocessor_opt = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01), ["Country", "Country Category"]),
            ("numerical", StandardScaler(), ["LOG_COMPLEXITY_WEIGHT_SUM"])
        ],
        remainder="passthrough",
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )

    model_opt = Pipeline(
        steps=[
            ("preprocessor", preprocessor_opt),
            ("regressor", build_linear_regressor(params)),
        ]
    )

    scores = cross_val_score(
        model_opt,
        X_train_opt,
        y_train_opt,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
    )

    return -scores.mean()

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=40, show_progress_bar=False)

best_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01), ["Country", "Country Category"]),
        ("numerical", StandardScaler(), ["LOG_COMPLEXITY_WEIGHT_SUM"])
    ],
    remainder="passthrough",
    sparse_threshold=0,
    verbose_feature_names_out=False,
)

best_model = Pipeline(
    steps=[
        ("preprocessor", best_preprocessor),
        ("regressor", build_linear_regressor(study.best_params)),
    ]
)

best_model.fit(X_train_opt, y_train_opt)
y_pred_opt = best_model.predict(X_test_opt)

optimized_results = {
    "Model": study.best_params["model_type"],
    "MAE": mean_absolute_error(y_test_opt, y_pred_opt),
    "MSE": mean_squared_error(y_test_opt, y_pred_opt),
    "RMSE": root_mean_squared_error(y_test_opt, y_pred_opt),
    "R2": r2_score(y_test_opt, y_pred_opt),
    "Best CV RMSE": study.best_value,
    "Best Params": study.best_params,
}

optimized_results

[I 2026-04-01 18:20:14,899] A new study created in memory with name: no-name-5e64d650-c259-4215-b799-d9d801d1f1f3
[I 2026-04-01 18:20:14,927] Trial 0 finished with value: 2.02530049895804 and parameters: {'model_type': 'Lasso', 'alpha': 0.39079671568228824}. Best is trial 0 with value: 2.02530049895804.
[I 2026-04-01 18:20:14,942] Trial 1 finished with value: 1.840776524378649 and parameters: {'model_type': 'Ridge', 'alpha': 15.741890047456641}. Best is trial 1 with value: 1.840776524378649.
[I 2026-04-01 18:20:14,956] Trial 2 finished with value: 2.150242336518687 and parameters: {'model_type': 'Lasso', 'alpha': 65.98711072054076}. Best is trial 1 with value: 1.840776524378649.
[I 2026-04-01 18:20:14,970] Trial 3 finished with value: 2.0247860318466024 and parameters: {'model_type': 'Ridge', 'alpha': 0.0012601639723276807}. Best is trial 1 with value: 1.840776524378649.
[I 2026-04-01 18:20:14,984] Trial 4 finished with value: 1.9364717786948091 and parameters: {'model_type': 'Lasso', 

{'Model': 'Lasso',
 'MAE': 1.6062891881042927,
 'MSE': 4.491621785454743,
 'RMSE': 2.119344659430066,
 'R2': 0.44810993867808036,
 'Best CV RMSE': 1.756220251237871,
 'Best Params': {'model_type': 'Lasso', 'alpha': 0.04578095241911155}}

### Stacking Regressor
---

In [451]:
from sklearn.ensemble import StackingRegressor

# Create our base estimators from the previously tuned parameters
estimators = [
    ('elasticnet', build_linear_regressor(study.best_params)),
    ('xgboost', xgb_model.named_steps['regressor'])
]

stacking_model = Pipeline(
    steps=[
        ("preprocessor", best_preprocessor),
        (
            "regressor",
            StackingRegressor(
                estimators=estimators,
                final_estimator=Ridge()
            ),
        ),
    ]
)

stacking_model.fit(X_train_opt, y_train_opt)
stacking_y_pred = stacking_model.predict(X_test_opt)

stacking_results = {
    "MAE": mean_absolute_error(y_test_opt, stacking_y_pred),
    "MSE": mean_squared_error(y_test_opt, stacking_y_pred),
    "RMSE": root_mean_squared_error(y_test_opt, stacking_y_pred),
    "R2": r2_score(y_test_opt, stacking_y_pred),
}

stacking_results

{'MAE': 1.6122717603114058,
 'MSE': 4.472260955158585,
 'RMSE': 2.1147720811374886,
 'R2': 0.45048882326137085}

### LightGBM
---

In [452]:
from lightgbm import LGBMRegressor

lgbm_model = Pipeline(
    steps=[
        ("preprocessor", best_preprocessor),
        (
            "regressor",
            LGBMRegressor(
                n_estimators=300,
                learning_rate=0.05,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            ),
        ),
    ]
)

lgbm_model.fit(X_train_opt, y_train_opt)
lgbm_y_pred = lgbm_model.predict(X_test_opt)

lgbm_results = {
    "MAE": mean_absolute_error(y_test_opt, lgbm_y_pred),
    "MSE": mean_squared_error(y_test_opt, lgbm_y_pred),
    "RMSE": root_mean_squared_error(y_test_opt, lgbm_y_pred),
    "R2": r2_score(y_test_opt, lgbm_y_pred),
}

lgbm_results

/Users/nickanthonymiras/miniconda3/envs/datascience/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


{'MAE': 1.6488336799587227,
 'MSE': 4.602501003190701,
 'RMSE': 2.145344029098993,
 'R2': 0.4344860982929022}

# Conclusion